# Extending the Embeddings Taxonomy Cross-Validation: Q4 and Q2

Continuation of the Q5 (Top-K) manual/builtin cross-validation, using the
same multi-seed methodology (10 random 70/30 splits, pooled confusion
matrix) already validated there -- see `ast_taxonomy_crossvalidation.ipynb`
and the paper's Section 5.6.

**Scope note:** this notebook only covers Q4 and Q2. Q1 and Q3 are
deliberately excluded -- both are single-strategy competencies
(`GuardedInput`/`LinearSearch`, `SelectionStep`) whose real finding was a
structural-matching *bug* (flipped comparisons, compound conditions), not a
taxonomic split between multiple legitimate strategies. There's no
analogous "does an independent method recover this same distinction"
question to ask for either of them.

**What this notebook checks, for each of Q4 and Q2:**
- **Q4**: does `codet5p-110m-embedding`, with no access to AST structure,
  independently recover the `BubbleSortPass` (13.1%) vs. `TopKPattern`
  (64.4%) split that makes up `SortByKey`?
- **Q2**: does it independently recover the counter-strategy (85.2%) vs.
  functional-pipeline (4.5%) split inside `UniqueCheck`? (The third real
  branch, `AccumulateThenCount`, is only ~0.7% of submissions -- too rare
  for a stable centroid, the same problem `novel` had in the Q5 experiment
  -- so it's dropped here, the same way `novel`'s instability was kept out
  of Q5's headline number.)

Requires: `topk_report.csv`-style CSVs for Q2 and Q4 (`q2_report.csv`,
`q4_report.csv`) and the real Refactory `.py` files for both exercises, all
in your Drive `PROJECT_DIR`, same layout as the other notebooks.


## 1. Setup

In [ ]:
!pip install -q transformers torch pandas scikit-learn matplotlib

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu (fine for codet5p-110m-embedding on short snippets)")


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/competency_extraction'  # <-- adjust if needed

import os
assert os.path.isdir(PROJECT_DIR), f"Not found: {PROJECT_DIR} -- check the folder name/path in Drive"
print("Contents:", os.listdir(PROJECT_DIR))


## 3. Load the embedding model

Same `codet5p-110m-embedding` setup as the other notebooks, config patch
included (works around the `is_decoder` compatibility bug without needing
any `transformers` version pin).


In [ ]:
from transformers import AutoConfig, AutoModel, AutoTokenizer
import torch

_checkpoint = "Salesforce/codet5p-110m-embedding"
_device = "cuda" if torch.cuda.is_available() else "cpu"

_tokenizer = AutoTokenizer.from_pretrained(_checkpoint, trust_remote_code=True)

_config = AutoConfig.from_pretrained(_checkpoint, trust_remote_code=True)
for _attr, _default in [("is_decoder", False), ("is_encoder_decoder", True),
                         ("pad_token_id", 0), ("eos_token_id", 2)]:
    if not hasattr(_config, _attr):
        setattr(_config, _attr, _default)

_model = AutoModel.from_pretrained(_checkpoint, config=_config, trust_remote_code=True).to(_device)
_model.eval()

@torch.no_grad()
def codebert_embed(code: str):
    inputs = _tokenizer.encode(code, return_tensors="pt", truncation=True, max_length=512).to(_device)
    embedding = _model(inputs)[0]
    return embedding.cpu().numpy()

def cosine_sim(a, b) -> float:
    import numpy as np
    return float(np.dot(a, b))

# Sanity check
related_a = "def f(lst):\n    best = lst[0]\n    for x in lst:\n        if x > best:\n            best = x\n    return best"
related_b = "def f(lst):\n    return max(lst)"
unrelated  = "def greet(name):\n    return 'Hello, ' + name"
print("related vs related:", cosine_sim(codebert_embed(related_a), codebert_embed(related_b)))
print("related vs unrelated:", cosine_sim(codebert_embed(related_a), codebert_embed(unrelated)))


## 4. Reusable multi-seed classification function

Same logic validated for Q5, factored out so it can run against any
two-label dataset without repeating the loop each time.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def run_multiseed_crossvalidation(labeled_df, embedding_cache, label_col="strategy_label",
                                   labels_order=None, n_seeds=10):
    """labeled_df must have columns: file, <label_col>. embedding_cache maps
    file -> embedding vector, already computed for every file in labeled_df.
    Returns (summary_df, pooled_df) -- summary_df has one row per seed,
    pooled_df has one row per (seed, file) prediction, pooled across all seeds.
    """
    if labels_order is None:
        labels_order = sorted(labeled_df[label_col].unique())

    per_seed_results = []
    all_predictions = []

    for seed in range(n_seeds):
        shuffled = labeled_df.sample(frac=1, random_state=seed).reset_index(drop=True)
        split_point = int(len(shuffled) * 0.7)
        train_df = shuffled.iloc[:split_point]
        test_df = shuffled.iloc[split_point:]

        centroids = {}
        for label in labels_order:
            files = train_df[train_df[label_col] == label]["file"].tolist()
            vecs = [embedding_cache[f] for f in files if f in embedding_cache]
            if vecs:
                centroids[label] = np.mean(vecs, axis=0)

        rows = []
        for _, row in test_df.iterrows():
            f = row["file"]
            if f not in embedding_cache:
                continue
            emb = embedding_cache[f]
            sims = {label: cosine_sim(emb, c) for label, c in centroids.items()}
            predicted = max(sims, key=sims.get)
            rows.append({"file": f, "seed": seed, "true_label": row[label_col], "predicted": predicted})

        results_s = pd.DataFrame(rows)
        all_predictions.append(results_s)
        acc = accuracy_score(results_s["true_label"], results_s["predicted"])
        per_seed_results.append({"seed": seed, "accuracy": acc, "n_test": len(results_s)})

    summary_df = pd.DataFrame(per_seed_results)
    pooled_df = pd.concat(all_predictions, ignore_index=True)

    print(summary_df)
    print(f"\naccuracy: mean={summary_df['accuracy'].mean():.3f}  std={summary_df['accuracy'].std():.3f}")

    cm = confusion_matrix(pooled_df["true_label"], pooled_df["predicted"], labels=labels_order)
    print(f"\npooled confusion matrix [rows=true, cols=predicted] ({len(pooled_df)} predictions across {n_seeds} seeds)")
    print(pd.DataFrame(cm, index=labels_order, columns=labels_order))
    print()
    print(classification_report(pooled_df["true_label"], pooled_df["predicted"], labels=labels_order))

    return summary_df, pooled_df


## 5. Q4: BubbleSortPass vs. TopKPattern

Direct analog of the Q5 manual/builtin check. Labels come straight from
`q4_report.csv`'s existing columns -- no elimination logic needed, since
both branches were already run as separate target IDs.


In [ ]:
q4_report = pd.read_csv(f"{PROJECT_DIR}/q4_report.csv")

def q4_label(row):
    if pd.to_numeric(row["BubbleSortPass_final"], errors="coerce") >= 0.999:
        return "bubblesort"
    elif pd.to_numeric(row["TopKPattern_final"], errors="coerce") >= 0.999:
        return "topk"
    return None  # neither matched -- the 22.4% "novel" group, excluded here

q4_report["strategy_label"] = q4_report.apply(q4_label, axis=1)
q4_labeled = q4_report[q4_report["strategy_label"].notna()][["file", "strategy_label"]]
print(q4_labeled["strategy_label"].value_counts())


In [ ]:
Q4_FOLDER = f"{PROJECT_DIR}/refactory/data/question_4/code"

def read_src(fname, folder):
    for sub in ["correct", "wrong"]:
        p = f"{folder}/{sub}/{fname}"
        if os.path.exists(p):
            return open(p, encoding="utf-8").read()
    return None

q4_embedding_cache = {}
for fname in q4_labeled["file"]:
    src = read_src(fname, Q4_FOLDER)
    if src is not None:
        q4_embedding_cache[fname] = codebert_embed(src)

print(f"cached {len(q4_embedding_cache)} of {len(q4_labeled)} Q4 files")


In [ ]:
q4_summary, q4_pooled = run_multiseed_crossvalidation(
    q4_labeled, q4_embedding_cache, label_col="strategy_label",
    labels_order=["bubblesort", "topk"], n_seeds=10,
)


## 6. Q2: counter vs. functional strategy

`AccumulateThenCount` (~0.7%) is dropped as too rare for a stable centroid,
the same reasoning that excluded `novel` from Q5's headline number. The
"functional" label is inferred by elimination: `UniqueCheck` matched, but
neither `ConditionalCount` nor `AccumulateThenCount` did.


In [ ]:
q2_report = pd.read_csv(f"{PROJECT_DIR}/q2_report.csv")

def q2_label(row):
    cc = pd.to_numeric(row["ConditionalCount_final"], errors="coerce")
    atc = pd.to_numeric(row["AccumulateThenCount_final"], errors="coerce")
    uc = pd.to_numeric(row["UniqueCheck_final"], errors="coerce")
    if cc >= 0.999:
        return "counter"
    if atc >= 0.999:
        return None  # accumulate-then-count: too rare (~0.7%), excluded
    if uc >= 0.999:
        return "functional"  # matched UniqueCheck via elimination -> functional branch
    return None  # unmatched -- the 9.6% long tail, excluded

q2_report["strategy_label"] = q2_report.apply(q2_label, axis=1)
q2_labeled = q2_report[q2_report["strategy_label"].notna()][["file", "strategy_label"]]
print(q2_labeled["strategy_label"].value_counts())


In [ ]:
Q2_FOLDER = f"{PROJECT_DIR}/refactory/data/question_2/code"

q2_embedding_cache = {}
for fname in q2_labeled["file"]:
    src = read_src(fname, Q2_FOLDER)
    if src is not None:
        q2_embedding_cache[fname] = codebert_embed(src)

print(f"cached {len(q2_embedding_cache)} of {len(q2_labeled)} Q2 files")


**Before running the classification below**, check the `value_counts()`
output from the cell above. `functional` is only 4.5% of the full Q2
corpus (~13 of 291 correct submissions) -- smaller than Q5's `novel` group
(48) and roughly Q1's `while`-condition gap (8) in absolute terms. Expect
higher seed-to-seed variance here than the Q4 or Q5 checks, for the same
reason `novel`'s recall was unstable in Q5 -- report the mean *and* the
spread, not a single number, and treat any headline accuracy from this
specific check with the same caution as Q5's `novel` recall got, not the
same confidence as Q4's or Q5's manual/builtin numbers.


In [ ]:
q2_summary, q2_pooled = run_multiseed_crossvalidation(
    q2_labeled, q2_embedding_cache, label_col="strategy_label",
    labels_order=["counter", "functional"], n_seeds=10,
)


## 7. Save results


In [ ]:
q4_pooled.to_csv(f"{PROJECT_DIR}/q4_embeddings_crossvalidation.csv", index=False)
q2_pooled.to_csv(f"{PROJECT_DIR}/q2_embeddings_crossvalidation.csv", index=False)

from google.colab import files
files.download(f"{PROJECT_DIR}/q4_embeddings_crossvalidation.csv")
files.download(f"{PROJECT_DIR}/q2_embeddings_crossvalidation.csv")
